## Metabolomics homework
### Ambrus Csaba

### LC–MS metabolomikai pipeline (v2, ST000816)

Lipid peak mátrix: neg + pos ionmód, minták × jellemzők. Lépések a `NMR_Metabolomika_Pipeline_v2.ipynb` szerkezetéhez igazítva: **QC → PQN/log1p/Pareto (+ klasszikus LC–MS előfeldolgozás) → multipanel PCA → univariáns + vulkán → heatmap → PLS-DA + VIP + S-plot → RF**. A függvények a `src/metabolomics_*.py` modulokban vannak.


In [ ]:
# Init gdrive and python environment

from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/csambrus/MetabolKD.git
%cd /content/MetabolKD
!pip install -r ./requirements.txt

In [ ]:
import sys
import traceback
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path("/content/MetabolKD")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
from src.runtime_setup import setup_tensorflow_runtime, setup_notebook_error_logger, set_global_seed

set_global_seed()
setup_notebook_error_logger()
setup_tensorflow_runtime()


## 1. Adatbetöltés (ST000816, baseline — mint a házi feladatban)


In [ ]:
from src.download_dataset import download_dataset, combine_pos_neg_to_feature_matrix

data = download_dataset()
X, meta = combine_pos_neg_to_feature_matrix(data)

print("NEG raw:", data["neg"].shape)
print("POS raw:", data["pos"].shape)
print("Feature matrix:", X.shape)
print(meta.head())

# Select only baseline samples — sorok szűrése: .loc (X[...] oszlopokat választana)
meta = meta[meta["visit"] == "Baseline"]
X = X.loc[meta.index]

# LC–MS pipeline: a további cellák X_raw néven hivatkoznak a mátrixra
X_raw = X


## 2. QC (mintánkénti összjel és hiányzás)


In [ ]:
from src.metabolomics_qc import qc_sample_table
from src.metabolomics_plotting import plot_qc_bars

qc = qc_sample_table(X_raw)
print(qc.describe())
fig = plot_qc_bars(qc, meta, hue_col="progressor_status")
fig.savefig("qc_overview.png", dpi=150, bbox_inches="tight")


## 3. NMR-stílusú mátrixok (PQN, log1p, Pareto) + klasszikus LC–MS előfeldolgozás


In [ ]:
from src.metabolomics_preprocessing import nmr_style_matrices, preprocess_feature_matrix

blocks = nmr_style_matrices(X_raw, impute_first=True)
for k, df in blocks.items():
    print(k, df.shape)
X_pareto = blocks["X_pareto"]
X_log = blocks["X_log"]

X_proc, steps = preprocess_feature_matrix(X_raw)
print("Klasszikus LC–MS lépések:", " → ".join(steps))
X_proc.head()


## 4. PCA multipanel (Pareto + log1p térben, extra Z-score nélkül — mint az NMR `X_PARETO`)


In [ ]:
from src.metabolomics_multivariate import fit_pca
from src.metabolomics_plotting import plot_pca_multipanel, plot_pca_scores

pca_out = fit_pca(X_pareto, n_components=5, standardize=False)
scores = pca_out["scores"]
var = pca_out["explained_variance_ratio"]
print("Magyarázott variancia (első 5 PC):", [round(float(v), 4) for v in var])

fig = plot_pca_multipanel(pca_out, meta, hue_col="progressor_status")
fig.savefig("pca_multipanel.png", dpi=150, bbox_inches="tight")

fig2 = plot_pca_scores(
    scores, meta,
    pc_x="PC1", pc_y="PC2",
    hue_col="progressor_status",
    explained=var,
    title="PCA score (Pareto-skálázott log1p)",
)
fig2.savefig("pca_scores.png", dpi=150, bbox_inches="tight")


## 5. Univariáns elemzés + vulkán (Baseline: Progressor vs Non-progressor)


In [ ]:
# Csak baseline látogatás
bl = meta["visit"] == "Baseline"
Xb_log = X_log.loc[bl]
Xb_par = X_pareto.loc[bl]
metab = meta.loc[bl]

from src.metabolomics_univariate import differential_analysis
from src.metabolomics_plotting import plot_volcano, plot_volcano_categorized

diff = differential_analysis(
    Xb_log,
    metab["progressor_status"],
    group_a="Progressor",
    group_b="Non-progressor",
    include_cohen_d=True,
)
print(diff.head(15))
fig = plot_volcano(diff, alpha=0.05, fc_thresh=0.5)
fig.savefig("volcano_baseline_progressor.png", dpi=150, bbox_inches="tight")
figc = plot_volcano_categorized(
    diff,
    group_up="Progressor",
    group_down="Non-progressor",
    alpha=0.05,
    fc_thresh=0.5,
)
figc.savefig("volcano_categorized.png", dpi=150, bbox_inches="tight")


## 6. Top eltérések heatmap


In [ ]:
from src.metabolomics_plotting import plot_top_features_heatmap

top_n = 25
top_feats = diff.nsmallest(top_n, "padj")["feature"].tolist()
fig = plot_top_features_heatmap(
    Xb_par,
    top_feats,
    metab,
    group_col="progressor_status",
    max_samples=50,
)
fig.savefig("heatmap_top_features.png", dpi=150, bbox_inches="tight")


## 7. PLS-DA + VIP + S-plot (Pareto mátrix)


In [ ]:
from src.metabolomics_multivariate import fit_plsda_multiclass
from src.metabolomics_plotting import plot_s_plot_lv1
import matplotlib.pyplot as plt

plsda = fit_plsda_multiclass(
    Xb_par,
    metab["progressor_status"],
    n_components=3,
    scale=True,
)
s_pls = plsda["scores"]
vip = plsda["vip"]
print(vip.nlargest(10))

fig, ax = plt.subplots(figsize=(7, 5))
for lab in metab["progressor_status"].unique():
    m = metab["progressor_status"] == lab
    ax.scatter(s_pls.loc[m, "LV1"], s_pls.loc[m, "LV2"], label=str(lab), s=45, alpha=0.85, edgecolors="white", linewidths=0.3)
ax.set_xlabel("LV1")
ax.set_ylabel("LV2")
ax.set_title("PLS-DA score (Progressor címke)")
ax.legend(title="progressor_status")
ax.axhline(0, color="gray", lw=0.4)
ax.axvline(0, color="gray", lw=0.4)
fig.tight_layout()
fig.savefig("plsda_scores.png", dpi=150, bbox_inches="tight")

w1 = plsda["model"].pls.x_weights_[:, 0]
figs = plot_s_plot_lv1(Xb_par, metab["progressor_status"], "Progressor", w1)
figs.savefig("splot_lv1.png", dpi=150, bbox_inches="tight")


## 8. Random Forest — feature importance (felügyelt, gyors)


In [ ]:
from src.metabolomics_ml import random_forest_feature_importance

imp = random_forest_feature_importance(
    Xb_par,
    metab["progressor_status"],
    "Progressor",
    n_estimators=300,
)
print(imp.head(20))


## 9. Rövid értelmezés

- **QC**: extrém `total_signal` vagy `missing_frac` minták kiszűrhetők.
- **PCA**: fő irányok a legnagyobb közös variancia mentén; színezés a klinikai címkével.
- **Vulkán + FDR**: egyszerre nézünk hatást (log2FC) és bizonytalanságot (p / FDR).
- **PLS-DA + VIP + S-plot**: felügyelt irányok és jellemző-súlyok (NMR notebook mintájára).
- **RF importance**: nemlineáris alternatíva a fontos peak-ekhez.

A `NMR_Metabolomika_Pipeline_v2.ipynb` referenciához igazítva: **PQN / log1p / Pareto**, **multipanel PCA**, **FDR (statsmodels BH)**, **kategorizált vulkán**, **PLS-DA + VIP**; GPU/SHAP részek nélkül, LC–MS lipid adatra.

